# Calculating Correlations and P-Values Between Legacy and Interim NetPP Products 

> History | Updated Dec 2024

## Objectives

This script generates correlations between two time series of monthly means of
primary productivity described by Behrenfeld and Falkowski 1997. The script
calculates Pearson correlations and can also be adapted for datasets
such as PAR, chlorophyll, SST, etc.

In this notebook, users will be able to:

* Generate correlations for satellite-derived monthly means of primary productivity.

* Choose between datasets representing raw data or anomalies.

* Customize inputs, time ranges, and output file settings. 

### Adapting the Script for Other Datasets

The script can be repurposed for different input data with these modifications:

1. Gridding any input chlorophyll, PAR, and SST data to a common grid.

2. Creating an appropriate cdl file for the dataset.

3. Adjusting the logic to make directory paths.

4. Adjust output file naming and input file search pattern.

## Tutorial for this notebook

By following this notebook, users will be able to:

* Generate correlations and p-values between the legacy product, **MODIS-AQUA**, and the interim product **VIIRS-SNPP** or **VIIRS-NOAA20**.

The result will be a NetCDF file populated with:

- **Pearson correlation coefficients** for each grid point.

- Corresponding **p-values** indicating statistical significance.

- The **number of valid data points** used for each calculation.

### Datasets Overview

1. **Primary Productivity, MODIS-AQUA, Science Quality, Global, 9km, 2013-2022 (Monthly Composite)**
- Distributed via the West Coast Node ERDDAP dataset at the following link:
> http://localhost:8080/erddap/griddap/productivity_modis_aqua_monthly.graph

2. **Primary Productivity, VIIRS-SNPP, Science Quality, Global, 9km, 2013-2022 (Monthly Composite)**
- Distributed via the West Coast Node ERDDAP dataset at the following link:
> http://localhost:8080/erddap/griddap/productivity_viirs_snpp_monthly.graph

3. **Primary Productivity, VIIRS-NOAA20, Science Quality, Global, 9km, 2018-2022 (Monthly Composite)**
- Distributed via the West Coast Node ERDDAP dataset at the following link:
> http://localhost:8080/erddap/griddap/productivity_viirs_noaa20_monthly.graph

### Resource requirements

- **Jupyter Notebook**

- **Python 3** with the modules included within the *Import packages* section below

- **The CDL file** located here: PUT GITHUB LINK
    * The cdl file has latitude and longitude that have been gridded to the 9km NOAA CoastWatch standard ocean color grid.

    * The template file is populated with the latitude and longitude data and then renamed.

- **Internet connection**

## Import packages

In [1]:
import os
import sys
import subprocess
from datetime import datetime
import numpy as np
import numpy.ma as ma
import pandas as pd
import xarray as xr
import xskillscore as xs
import netCDF4

## Create global variables
Global variables are used to set up the file structure, directory paths, and template files required for processing correlations.

* **BASE_DIR** is the base directory defined for the project, and the root for all other subdirectories.
* **WORK_DIR** is where the temporary files used during data processing are located.
* **RESOURCES_DIR** is where the CDL file will be located.
* **CORRELATIONS_DIR** is where the output correlations file will be located.

In [2]:
BASE_DIR = '/Users/madisonrichardson/netpp'
WORK_DIR = os.path.join(BASE_DIR, 'work')
RESOURCES_DIR = os.path.join(BASE_DIR, 'resources')
CORRELATIONS_DIR = os.path.join(BASE_DIR, 'data', 'correlations')

## Create some useful functions
### Function to validate parameters
The function validates input parameters to ensure they fall within correct ranges and maintain logical order. 

In [3]:
def validate_params(startyear, endyear, percent_keep):
    """
    Validates the inputted parameters to ensure
    correct ranges and logical order.

    Args:
        startyear (str): The start year in 'YYYYMM' format.
        endyear (str): The end year in 'YYYYMM' format.
        percent_keep (float): Percentage of grid points to keep (0 to 1)

    Raises:
        ValueError: The startyear is not earlier than endyear.
        ValueError: The percent_keep is outside the range (0 to 1).
    """
    if startyear > endyear:
        raise ValueError("The start year must be earlier than the end year.")
    if not (0.0 <= percent_keep <= 1.0):
        raise ValueError("percent_keep must be between 0 and 1")

### Function to parse dates
This function extracts year and month components from start and end dates provided in the 'YYYYMM' format.

In [4]:
def parse_dates(startyear, endyear):
    """
    Parses the start and end years in 'YYYYMM' format
    and extracts the corresponding years and months as
    separate integer values.
 
    Args:
        startyear (str): The start date in 'YYYYMM' format.
        endyear (str): The end date in 'YYYYMM' format.

    Returns:
        tuple: A tuple containing four integers:
        (syear, smonth, eyear, emonth).
            - syear (int): The year component of the start date.
            - smonth (int): The month component of the start date.
            - eyear (int): The year component of the end date.
            - emonth (int): The monht component of the end date.
    """
    syear = int(np.floor(int(startyear) / 100))
    smonth = int(startyear) - syear * 100
    eyear = int(np.floor(int(endyear) / 100))
    emonth = int(endyear) - eyear * 100
    return syear, smonth, eyear, emonth

### Function to extract ERDDAP data

This function retrieves and opens a dataset from an ERDDAP server using 'xarray'.

In [5]:
def xr_open_ds(e_id, e_source='http://localhost:8080/erddap', dap='griddap', var_name=None):
    """
    Open a remote ERDDAP dataset by constructing
    the full URL for the specified dataset and opens it as
    an xarray Dataset. If a variable name is provided, it attempts
    to extract and return only that variable as an xarray
    DataArray.

    Args:
        e_id (str): The dataset ID for the dataset on the ERDDAP server.
        e_source (str): The base URL of the ERDDAP server. Defaults to
        'http://localhost:8080/erddap'.
        dap (str): The data acess protocol to us (e.g., 'griddap').
        Defaults to 'griddap'.
        var_name (str, optional): The name of the variable to extract from
        the dataset. If None, the entire dataset is returned.

    Returns:
        xarray.Dataset or xarray.DataArray:
            - The entire dataset if 'var_name' is None.
            - The specified variable as a DataArray if 'var_name' is returned.

    Raises:
        KeyError: If the specified variable name does not exist in the dataset.
    """
    # remove any trailing /
    e_source = e_source.rstrip("/")

    erddap_url = '/'.join([e_source,
                           dap,
                           e_id
                           ])
    
    ds = xr.open_dataset(erddap_url)

    if var_name:
        if var_name in ds:
            return ds[var_name]
        else:
            print(f"Variable '{var_name}' not found in the dataset.")
            print("Available variables:")
            print(list(ds.variables.keys()))
            raise KeyError(f"Variable  '{var_name}' not found.")

    return ds

### Function to reshape data into blocks
This function partitions the latitude dimension of a 2D array into smaller, manageable segments to optimize data processing efficiency.

In [6]:
def reshape_data_block(data, block_size):
    """
    Reshapes a 2D array's latitude dimension into
    smaller blocks for efficient processing.

    Args:
        data (np.ndarray): A 2D array with shape (time, latitude, longitude)
        where the latitude dimension (axis=1) will be divide into blocks.
        block_size (int): The number of latitude grid points in each block.

    Returns:
        np.ndarray: A 2D array where the latitude indices are reshaped
        into blocks of the specified size, with shape (num_blocks, block_size).
    """
    ny_index = np.arange(data.shape[1])
    num_blocks = int(data.shape[1] / block_size)
    return np.reshape(ny_index, [num_blocks, block_size])

## Set Up Parameters and Validate Inputs

Initialize the parameters required for the correlation calculation:

- **startyear** and **endyear**: Define the date range for the analysis (format: YYYYMM).

- **source1** and **source2**: Specify the data sources for comparison ('modis' & 'noaa20' in this case)

- **ncvar**: The name of the variable being analyzed (we are using 'productivity')

- **timeseries_type**: Indicated whether raw data ('data') or anomalies ('anom') will be used.

- **percent_keep**: The minimum proportion of non-missing data requires for correlation calculations

- **overwrite**: If 'True', exisiting output files will be replaced.

- **ny_block**: Number of latitude grid points per block processing.

The 'validate_params' function ensures that the inputs are valid.

In [7]:
# Set parameters
startyear = "201801"
endyear = "202212"
source1 = "modis"
source2 = "noaa20"
ncvar = "productivity"
timeseries_type = "data"
percent_keep = 0.5
overwrite = True
ny_block = 20

# Validate parameters
try:
    validate_params(startyear, endyear, percent_keep)
except ValueError as e:
    print(f"Parameter validation error: {e}")
    sys.exit(1)

## Parse Dates and Generate Time Series

Convert 'startyear' and 'endyear' into a range of monthly timestamps.

-  A list of dates ('dtM') and corresponding year ('yy') and month ('mm') arrays.

- Uses 'np.datetime64' and 'pandas' to create a monthly time range.

- Calculates the total number of months ('ntM') for looping through data.

In [8]:
# Parse and validate inputs
syear, smonth, eyear, emonth = parse_dates(startyear, endyear)

# Generate monthly date range from the start to the end
time_bgn = np.datetime64('{}-{:02d}'.format(syear, smonth), 'M')
time_end = np.datetime64('{}-{:02d}'.format(eyear, emonth), 'M')
dtM = pd.to_datetime(np.arange(time_bgn, time_end+1, dtype='datetime64[M]'))
ntM = len(dtM)
yy = dtM.year.values
mm = dtM.month.values

print(f"Generated {ntM} monthly time points from {time_bgn} to {time_end}")

Generated 60 monthly time points from 2018-01 to 2022-12


## Directory Set Up

Ensure that all required directories for input and output files exist.

- The list 'DIR_LIST' contains paths to essential directories

- Each directory is created if it does not already exist using 'os.makedirs'.

- Outputs the number of validated directories.

In [9]:
# Verify directories exist
DIR_LIST = [
    BASE_DIR,
    WORK_DIR,
    RESOURCES_DIR,
    CORRELATIONS_DIR
]

for dr in DIR_LIST:
    os.makedirs(dr, exist_ok=True)
print(len(DIR_LIST), 'directories validated')

4 directories validated


## Prepare Input and Output Templates

Define file naming conventions for the output file.

- **ofile_tmpl**: Template for output file names (correlation results).

- Skips processing if the output file already exists and 'overwrite' is 'False'. 

In [10]:
# Set up the template output file
ofile_tmpl = '{}_{}_corr_month_{}_{}_{}_{}_{}_{:03d}percent.nc'

# Calculate correlations for ncvar variable wanted
ncvar_list = [ncvar]
for ncvar in ncvar_list:
    # Set up directories and output file parameters
    source_list = np.sort([source1, source2])
    source_comb = '{}_{}'.format(source_list[0], source_list[1])
    odir = os.path.join(
        CORRELATIONS_DIR,
        source_comb
    )
    odir_filelist = os.listdir(odir)

    # Generate output filenaeme
    ofile = ofile_tmpl.format(
        ncvar,
        timeseries_type,
        source_list[0],
        source_list[1],
        '9km',
        startyear,
        endyear,
        int(percent_keep*100)
    )

    # Skip if output file exists and overwrite is False
    if not overwrite and ofile in odir_filelist:
        print(f'{ofile} already exists')
        continue

## Prepare and Run ncgen Command

Generate an initial NetCDF file using the 'ncgen' tool and a CDL template.

- Creates filenames for the output NetCDF ('ncgen_ofile_nc') and CDL input ('ncgen_ifile_cdl').

- Constructs a shell command for 'ncegn' to convert the CDL file into a NetCDF file.

- Executes the command using 'subprocess.call' and checks for errors.

In [11]:
# Prepare ncgen input and output filenames
now = datetime.now()
ncgen_ofile_nc = f"ncgen_corr_ofile{now:%Y%m%d%H%M%S}.nc"
ncgen_ifile_cdl = 'correlations_{}_month_{}_{}_{}.cdl'.format(
    ncvar,
    source_list[0],
    source_list[1],
    '9km'
)

# Run ncgen command
myCmd1 = ' '.join(['ncgen',
                    '-o',
                    os.path.join(WORK_DIR, ncgen_ofile_nc),
                    os.path.join(RESOURCES_DIR, ncgen_ifile_cdl)
                    ])
print(myCmd1)
print('ncgen', subprocess.call(myCmd1, shell=True))

ncgen -o /Users/madisonrichardson/netpp/work/ncgen_corr_ofile20250226132530.nc /Users/madisonrichardson/netpp/resources/correlations_productivity_month_modis_noaa20_9km.cdl
ncgen 0


## Fetching MODIS and NOAA20 Datasets from ERDDAP

Using the **xr_open_ds** function we defined above, we will be retrieving the MODIS and NOAA20 data from ERDDAP. 

Set the base URL and dataset IDs for accessing MODIS and NOAA20 primary productivity data from an ERDDAP server.

It is optional to specify a variable, but in this case we are specifying the variable **productivity** which we define earlier as **ncvar**.

**If using SNPP**, change one of the sources to **snpp_id = "productivity_viirs_snpp_monthly"** and update to **snpp_ds**.

In [12]:
# For ERDDAP function
erddap_url = "http://localhost:8080/erddap"
modis_id = "productivity_modis_aqua_monthly"
noaa20_id = "productivity_viirs_noaa20_monthly"

modis_ds = xr_open_ds(
    e_source=erddap_url,
    e_id=modis_id,
    var_name=ncvar
)
print("MODIS Dataset:", modis_ds)

noaa20_ds = xr_open_ds(
    e_source=erddap_url,
    e_id=noaa20_id,
    var_name=ncvar
)
print("NOAA20 Dataset:", noaa20_ds)

MODIS Dataset: <xarray.DataArray 'productivity' (time: 120, latitude: 2160, longitude: 4320)> Size: 4GB
[1119744000 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 960B 2013-01-16T08:00:00 ... 2022-12-16T...
  * latitude   (latitude) float64 17kB 89.96 89.88 89.79 ... -89.88 -89.96
  * longitude  (longitude) float64 35kB -180.0 -179.9 -179.8 ... 179.9 180.0
Attributes:
    colorBarMaximum:        30.0
    colorBarMinimum:        0.0
    coverage_content_type:  modelResult
    ioos_category:          Other
    long_name:              Net Primary Productivity Monthly Mean
    source:                 NOAA CoastWatch West Coast Node
    standard_name:          net_primary_productivity
    units:                  mg C m-2 d-1
NOAA20 Dataset: <xarray.DataArray 'productivity' (time: 60, latitude: 2160, longitude: 4320)> Size: 2GB
[559872000 values with dtype=float32]
Coordinates:
  * time       (time) datetime64[ns] 480B 2018-01-16T12:00:00 ... 2022-12-16T...
  * 

## Preparing Monthly Timeseries from ERDDAP Datasets

Process the MODIS and NOAA20 data to extract monthly time slices and store them in a structured list for further analysis.

In [13]:
# Intialize an empty list to store datasets
ds_list = []

# ERDDAP Datasets
datasets = [modis_ds, noaa20_ds]

for ds in datasets:
    ds_dates_list = []  # To store individual monthly slices
    for i in range(ntM):
        # Select a single time slice
        start_date = f"{yy[i]}-{mm[i]:02d}-16"
        try:
            # Extract data for the specific time slice
            time_slice = ds.sel(time=start_date)
            ds_dates_list.append(time_slice)
        except KeyError as e:
            print(f"Time {start_date} not found in dataset: {e}")
            continue

    # Append the time series for the dataset
    ds_list.append(ds_dates_list)

print(f"Loaded data for {len(ds_list)} datasets.")

Loaded data for 2 datasets.


## Intializing matrices

Initialize matrices to store correlation coefficients, p-values, and the number of valid data points for each grid point, and extracts the dimensions of the data.

In [14]:
# Intialize matrices for correlations and p-values
_, ny1, nx1 = ds_list[0][0].shape
nt1 = len(ds_list[0])

print(f"Temporal Dimension (nt1): {nt1}")
print(f"Latitude Dimension (ny1): {ny1}, Longitude dimensions (nx1): {nx1}")

# Intialize matrices with NaN values
corr_mtrx = np.zeros([ny1, nx1])*np.nan
pval_mtrx = np.zeros([ny1, nx1])*np.nan
n_mtrx = np.zeros([ny1, nx1])*np.nan

Temporal Dimension (nt1): 60
Latitude Dimension (ny1): 2160, Longitude dimensions (nx1): 4320


# Reshaping latitude data

Reshapes the latitude dimension of the dataset into smaller blocks for efficient processing. 

This is useful when using larger datasets where processing the entire latitude range at once might not be feasible. 

In [15]:
# Reshape data into latitude blocks
indx_block = reshape_data_block(
    data=ds_dates_list[0].values,
    block_size=ny_block
)

# Calculate the number of latitude blocks
num_block = indx_block.shape[0]

print(f"Latitude divided into {num_block} blocks of size {ny_block}")

Latitude divided into 108 blocks of size 20


## Computing correlations, p-values, and valid counts

Process the dataset block by block, calculating correlations, p-values, and valid data counts for grid points in each latitude block. 

This chunk takes **around 8 minutes** to run. 

In [16]:
for i in range(num_block):
    print(f"Processing block {i + 1}/{num_block}")
    # Construct the two data matrix of shape [ntM X ny_block X nx1]
    injk = indx_block[i, :]
    data_block_mtrx = np.zeros([len(ds_list), ntM, ny_block, nx1])

    # Populate data_block_mtrx with data from each source and time step
    for j in range(ntM):
        for k in range(len(ds_list)):
            data_block_mtrx[k, j, :, :] = ds_list[k][j].values[0, injk, :]

    # Compute correlations for each latitude subset in the block
    for j in range(ny_block):
        data_mtrx = data_block_mtrx[:, :, j, :]

        # Only keep grid points that have non-missing numbers above 'percent_keep'
        ones_mtrx = data_mtrx/data_mtrx
        ones_mtrx_comb = np.sum(ones_mtrx, axis=0)/2
        sum_one_mtrx = np.nansum(ones_mtrx_comb, axis=0)
        in_keep = np.where(sum_one_mtrx > ntM*percent_keep)[0]

        # Create data array with time as coordinate, useful for
        # Calculating anoms in xarray
        time_series_data = xr.DataArray(
            data_mtrx[:, :, in_keep],
            coords=[
                source_list,
                dtM.astype('datetime64[ns]'),
                in_keep
            ],
            dims=['source', 'time', 'in_keep'])

        # Correlations for either data or anom time series
        if timeseries_type == 'anom':
            # Calculate monthly climatology
            monthly_climatology = time_series_data.groupby('time.month').mean('time')
            # Get anomalies by subtracting climatology from the data
            compare_data = time_series_data.groupby('time.month') - monthly_climatology
        elif timeseries_type == 'data':
            # Use raw data directly for correlation
            compare_data = time_series_data

        # Calculate Pearson correlation between sources
        correlation = xr.corr(
            compare_data.sel(source=source_list[0]),
            compare_data.sel(source=source_list[1]),
            dim='time'
        )

        # Use xskillscore to get p values
        p_value = xs.pearson_r_p_value(
            compare_data.sel(source=source_list[0]),
            compare_data.sel(source=source_list[1]),
            dim='time',
            skipna=True
        )

        # Place corr, pval values for latitude subset in final global data matrix
        corr_mtrx[injk[j], in_keep] = correlation.data
        pval_mtrx[injk[j], in_keep] = p_value.data
        # Store the count of valid data points
        n_mtrx[injk[j], in_keep] = sum_one_mtrx[in_keep]

print("Correlations and p-values computed.")
print(f"Min: {np.nanmin(corr_mtrx)}, Max: {np.nanmax(corr_mtrx)}, Mean: {np.nanmean(corr_mtrx)}")
print(f"Min: {np.nanmin(pval_mtrx)}, Max: {np.nanmax(pval_mtrx)}, Mean: {np.nanmean(pval_mtrx)}")
print(f"Min: {np.nanmin(n_mtrx)}, Max: {np.nanmax(n_mtrx)}, Mean: {np.nanmean(n_mtrx)}")

Processing block 1/108
Processing block 2/108
Processing block 3/108
Processing block 4/108
Processing block 5/108
Processing block 6/108
Processing block 7/108
Processing block 8/108
Processing block 9/108
Processing block 10/108
Processing block 11/108
Processing block 12/108
Processing block 13/108
Processing block 14/108
Processing block 15/108
Processing block 16/108
Processing block 17/108
Processing block 18/108
Processing block 19/108
Processing block 20/108
Processing block 21/108
Processing block 22/108
Processing block 23/108
Processing block 24/108
Processing block 25/108
Processing block 26/108
Processing block 27/108
Processing block 28/108
Processing block 29/108
Processing block 30/108
Processing block 31/108
Processing block 32/108
Processing block 33/108
Processing block 34/108
Processing block 35/108
Processing block 36/108
Processing block 37/108
Processing block 38/108
Processing block 39/108
Processing block 40/108
Processing block 41/108
Processing block 42/108
P

## Mask the invalid values
Mask NaN values in correlation, p-value, and count matrices to ensure that subsequent computations handle these cases properly.

In [17]:
# Mask invalid values in correlation, p-value, and count matrices
corr_mtrx = ma.masked_invalid(corr_mtrx)
pval_mtrx = ma.masked_invalid(pval_mtrx)
n_mtrx = ma.masked_invalid(n_mtrx)

## Save Results to a NetCDF file

Store correlation and p-value matrices in a NetCDF file for further analysis.

- Open the temporary NetCDF file created by 'ncgen' in append mode.

- Add global attributes to describe the dataset.

- Save correlation (**corr**), p-value (**pval**), and valid data count (**n**) matrices to the file.

- Use 'nccopy' for compression and save the final output file.

- Delete the temporary file to clean up.

### Final Output

The compressed NetCDF file contains:

- **corr**: Pearson correlation coefficients for each grid point.

- **pval**: Corresponding p-values indicating statistical significance.

- **n**: Number of valid data points used for each calculation.

The output filename follows the 'ofile_tmpl' convention.

Outputs confirmations of successfull processing and file saving.

In [18]:
# Labels and matrices to save in NetCDF file
cpn_lbl = ['corr', 'pval', 'n']
cpn_list = [corr_mtrx, pval_mtrx, n_mtrx]

# Open temporary file and load data into it
with netCDF4.Dataset(os.path.join(WORK_DIR, ncgen_ofile_nc), 'a') as nc:
    # Set global attributes
    nc.acknowledgement = (
        f"The project was supported by funding from the "
        f"Portfolio Management Branch of NESDIS and NOAA CoastWatch."
    )
    nc.contributors = (
        f"Dale Robinson, Isaac Shroeder, Ryan Vandermeulen, "
        "Jonathan Sherman, Jesse Espinoza, & Madison Richardson"
    )
    nc.title = (
        f"Pearson correlation coefficients and p-values of "
        f"monthly primary productivity fields between "
        f"{source1.upper()} and {source2.upper()}"
        )
    nc.summary = (
        f"Correlations between primary productivity or "
        f"PAR, chlorophyl and SST from {source1.upper()} "
        f"and {source2.upper()}. These are 9km products "
        f"generated from time series of monthly means. "
        f"See Melin et al 2017 for more details")
    nc.source = f"{source1.upper()} and {source2.upper()}"
    nc.instrument = f"{source1.upper()} and {source2.upper()}"
    nc.id = f"correlations_{source1}_{source2}_monthly_9km"
    nc.platform = f"{source1.upper()} and {source2.upper()}"
    
    # Place corr, pval, n in nc
    for j in range(len(cpn_lbl)):
        nc['{}'.format(cpn_lbl[j])][0, :, :] = cpn_list[j]

# Compress and save the temporary file to the final file name
myCmd = ' '.join(['nccopy',
                    '-d6',
                    os.path.join(WORK_DIR, ncgen_ofile_nc),
                    os.path.join(odir, ofile)
                    ])
print('nccopy', subprocess.call(myCmd, shell=True))
print('Done with', ofile)

# Clean up temporary file after saving the final output
os.remove(os.path.join(WORK_DIR, ncgen_ofile_nc))

/var/folders/81/qj7mv_yn7p98wpb9n0np6q8c0000gn/T/ipykernel_30139/3861596523.py:34: DeprecationWarning: `in1d` is deprecated. Use `np.isin` instead.
  nc['{}'.format(cpn_lbl[j])][0, :, :] = cpn_list[j]


nccopy 0
Done with productivity_data_corr_month_modis_noaa20_9km_201801_202212_050percent.nc
